# Kalman Filter — Two-Generator Power System with Line Trip

| | |
|---|---|
| **Author** | Marin Dagron |
| **Date** | January 2026 |
| **Context** | Industrial Project — CentraleSupélec |

---

## Objective

This notebook implements a **discrete-time Kalman Filter** to reconstruct the full dynamic state of a two-generator power system subject to a **transmission line trip at $t = 2\,\text{s}$**, which halves the synchronising coefficient $K_L$.

**Load disturbances are set to zero** — both local loads remain constant at their nominal values throughout.

The approach follows four main steps:

1. **Nonlinear simulation** — split at the fault instant; ground-truth using the full nonlinear model.
2. **Linearization** — two sets of state-space matrices: one **pre-fault** ($K_L$) and one **post-fault** ($K_L/2$).
3. **Observability analysis** — both models are checked.
4. **Kalman filtering** — the predict/correct loop **switches** its linear model at $t = 2\,\text{s}$, and a bias correction $\mathbf{b} = f_{\text{post}}(\mathbf{x}_{eq})$ is added to the prediction step.

---

## Notebook Structure

```
0. Imports & Configuration
1. System Parameters
2. Nonlinear Model & Simulation  ← line trip at t = 2 s, no load disturbance
3. Linearization — Pre-fault and Post-fault Models
4. Observability Analysis
5. Linear Model Simulation (Euler, switching)
6. Kalman Filter (switching model at t = 2 s)
7. Results & Visualisation
```

---
## 0. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from scipy.integrate import solve_ivp
from scipy.linalg import expm
from tqdm import tqdm

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (11, 4),
    'axes.grid': True,
    'grid.alpha': 0.4,
    'lines.linewidth': 1.5,
    'font.size': 11,
})

print('All imports successful.')

---
## 1. System Parameters

All physical parameters are centralised in a single dictionary.

Two additional parameters are introduced for the **line trip** scenario:

| Symbol | Description | Unit |
|--------|-------------|------|
| $t_{\text{break}}$ | Fault instant (line trip) | s |
| $K_L^{\text{post}}$ | Post-fault synchronising coefficient ($= K_L / 2$) | MW/rad |

**Load disturbance:** $\Delta p_{L1} = \Delta p_{L2} = 0$ — loads are constant throughout.

In [ ]:
PARAMS = {
    # Network
    'f0'     : 50.0,
    'omega0' : 2 * np.pi * 50.0,   # rad/s
    'KL'     : 3064.0,             # MW/rad  (pre-fault: two parallel lines)
    'KL_post': 1532.0,             # MW/rad  (post-fault: KL/2, one line tripped)
    't_break': 2.0,                # fault instant [s]
    'Ks'     : 0.05,               # secondary control gain

    # Generator 1
    'g1': {
        'P0'  : 600.0,   # nominal active power setpoint   [MW]
        'Pmin': 0.0,
        'Pmax': 1000.0,
        'Pr'  : 100.0,   # secondary participation factor  [MW]
        'J'   : 0.4,     # inertia constant
        'D'   : 0.04,    # damping coefficient
        'alpha': 100.0,  # governor proportional gain
        'beta' : 2000.0, # governor frequency gain
        'PL'  : 400.0,   # local load                      [MW]
    },

    # Generator 2
    'g2': {
        'P0'  : 400.0,
        'Pmin': 0.0,
        'Pmax': 600.0,
        'Pr'  : 50.0,
        'J'   : 0.1,
        'D'   : 0.02,
        'alpha': 100.0,
        'beta' : 2000.0,
        'PL'  : 600.0,
    },

    # Simulation
    'T_NL': 20.0,   # total simulation duration [s]
}

KL_pre  = PARAMS['KL']
KL_post = PARAMS['KL_post']
T_BREAK = PARAMS['t_break']
T_NL    = PARAMS['T_NL']
w0      = PARAMS['omega0']

print(f"Nominal frequency : {PARAMS['f0']} Hz")
print(f"Pre-fault KL      : {KL_pre} MW/rad")
print(f"Post-fault KL     : {KL_post} MW/rad  (= KL / 2)")
print(f"Line trip at      : t = {T_BREAK} s")
print(f"Load disturbance  : NONE  (PL1 = {PARAMS['g1']['PL']} MW, PL2 = {PARAMS['g2']['PL']} MW, constant)")

---
## 2. Nonlinear Model & Simulation

### State vector

$$\mathbf{x} = [\theta_1,\; \omega_1,\; T_{m1},\; \theta_2,\; \omega_2,\; T_{m2},\; N]^\top$$

### Line trip

The synchronising coefficient switches at the fault instant:

$$K_L(t) = \begin{cases} K_L = 3064\;\text{MW/rad} & t < 2\,\text{s} \\ K_L/2 = 1532\;\text{MW/rad} & t \geq 2\,\text{s} \end{cases}$$

The ODE is solved in **two consecutive segments** (pre- and post-fault) to handle the discontinuity cleanly:
- $[0,\; 2)$ s with $K_L$, initial condition = pre-fault equilibrium
- $[2,\; 20]$ s with $K_L/2$, initial condition = state at end of first segment

### No load disturbance

Loads are fixed: $p_{L1} = 400\;\text{MW}$, $p_{L2} = 600\;\text{MW}$.  
The disturbance input $\Delta\mathbf{w} = 0$.

In [ ]:
def power_system_ode(t: float, x: list, params: dict, kl_val: float) -> list:
    """
    Full nonlinear ODE of the two-generator power system.

    State  : x = [theta1, omega1, Tm1, theta2, omega2, Tm2, N]
    kl_val : current line synchronising coefficient [MW/rad].
             The caller is responsible for passing the correct value
             (pre-fault or post-fault).

    Load disturbance: ZERO — loads are constant at their nominal values.

    Returns dx/dt.
    """
    theta1, omega1, Tm1, theta2, omega2, Tm2, N = x
    w0  = params['omega0']
    p1l = params['g1']['PL']   # 400 MW — constant, no disturbance
    p2l = params['g2']['PL']   # 600 MW — constant, no disturbance

    # ── Algebraic equations ───────────────────────────────────────────────────
    F12  = kl_val * np.sin(theta1 - theta2)   # nonlinear tie-line power [MW]
    P1G  = p1l + F12                           # demand seen by G1
    P2G  = p2l - F12                           # demand seen by G2
    P1m  = Tm1 * omega1                        # mechanical power G1
    P2m  = Tm2 * omega2                        # mechanical power G2

    P1c = np.clip(params['g1']['P0'] + N * params['g1']['Pr'],
                  params['g1']['Pmin'], params['g1']['Pmax'])
    P2c = np.clip(params['g2']['P0'] + N * params['g2']['Pr'],
                  params['g2']['Pmin'], params['g2']['Pmax'])

    # ── Differential equations ────────────────────────────────────────────────
    d_theta1 = omega1 - w0
    d_omega1 = (Tm1 - P1G / omega1 - params['g1']['D'] * (omega1 - w0)) / params['g1']['J']
    d_Tm1    = (-params['g1']['alpha'] * (P1m - P1c) - params['g1']['beta'] * (omega1 - w0)
                if params['g1']['Pmin'] <= P1m <= params['g1']['Pmax'] else 0.0)

    d_theta2 = omega2 - w0
    d_omega2 = (Tm2 - P2G / omega2 - params['g2']['D'] * (omega2 - w0)) / params['g2']['J']
    d_Tm2    = (-params['g2']['alpha'] * (P2m - P2c) - params['g2']['beta'] * (omega2 - w0)
                if params['g2']['Pmin'] <= P2m <= params['g2']['Pmax'] else 0.0)

    omega_avg = (params['g1']['J'] * omega1 + params['g2']['J'] * omega2) /                 (params['g1']['J'] + params['g2']['J'])
    dN = -params['Ks'] * (omega_avg - w0) if -1.0 <= N <= 1.0 else 0.0

    return [d_theta1, d_omega1, d_Tm1, d_theta2, d_omega2, d_Tm2, dN]

In [ ]:
# ── Pre-fault equilibrium initial conditions ──────────────────────────────────
theta2_eq = -np.arcsin((PARAMS['g1']['P0'] - PARAMS['g1']['PL']) / KL_pre)

x0_nl = [
    0.0,                               # theta1  (bus 1 is angle reference)
    w0,                                # omega1
    PARAMS['g1']['P0'] / w0,           # Tm1
    theta2_eq,                         # theta2
    w0,                                # omega2
    PARAMS['g2']['P0'] / w0,           # Tm2
    0.0,                               # N
]

F12_eq = KL_pre * np.sin(x0_nl[0] - x0_nl[3])
print("Pre-fault equilibrium:")
print(f"  theta1 = {x0_nl[0]:.6f} rad")
print(f"  theta2 = {x0_nl[3]:.6f} rad  ({np.degrees(x0_nl[3]):.3f}°)")
print(f"  F12    = {F12_eq:.2f} MW  (= P01 - PL1 = {PARAMS['g1']['P0']-PARAMS['g1']['PL']:.0f} MW ✓)")
print(f"  Tm1    = {x0_nl[2]:.6f},  Tm2 = {x0_nl[5]:.6f}")

# ── Solver options ────────────────────────────────────────────────────────────
ODE_OPTS = dict(method='RK45', rtol=1e-8, atol=1e-10)
DT_EVAL  = 1e-4   # output sampling [s]

# ── Segment 1 : pre-fault  [0, t_break] ──────────────────────────────────────
t_eval_pre = np.arange(0.0, T_BREAK + DT_EVAL * 0.5, DT_EVAL)
sol_pre = solve_ivp(
    lambda t, x: power_system_ode(t, x, PARAMS, KL_pre),
    (0.0, T_BREAK), x0_nl,
    t_eval=t_eval_pre, **ODE_OPTS
)
print(f"\nPre-fault  : {sol_pre.message}  ({sol_pre.t.size} output points)")

# ── Segment 2 : post-fault [t_break, T_NL] ───────────────────────────────────
x_break = sol_pre.y[:, -1]          # state at the fault instant (continuity)
# t_eval_post = np.arange(T_BREAK, T_NL + DT_EVAL * 0.5, DT_EVAL)
num_points = int(round((T_NL - T_BREAK) / DT_EVAL)) + 1
t_eval_post = np.linspace(T_BREAK, T_NL, num_points)
sol_post = solve_ivp(
    lambda t, x: power_system_ode(t, x, PARAMS, KL_post),
    (T_BREAK, T_NL), x_break,
    t_eval=t_eval_post, **ODE_OPTS
)
print(f"Post-fault : {sol_post.message}  ({sol_post.t.size} output points)")

# ── Concatenate — drop the duplicate at t_break ───────────────────────────────
t_nl = np.concatenate([sol_pre.t, sol_post.t[1:]])
y_nl = np.concatenate([sol_pre.y, sol_post.y[:, 1:]], axis=1)
N_NL = len(t_nl)
print(f"\nTotal nonlinear solution: {N_NL} points over [0, {T_NL}] s")

In [ ]:
# ── KL array: correct coefficient at every time step ─────────────────────────
kl_arr = np.where(t_nl < T_BREAK, KL_pre, KL_post)

# ── Nonlinear derived signals ─────────────────────────────────────────────────
F12_nl = kl_arr * np.sin(y_nl[0] - y_nl[3])   # exact nonlinear tie-line power
P1L    = PARAMS['g1']['PL']                    # 400 MW (constant)
P2L    = PARAMS['g2']['PL']                    # 600 MW (constant)

STATE_COLS  = ['theta1', 'omega1', 'Tm1', 'theta2', 'omega2', 'Tm2', 'N']
OUTPUT_COLS = ['theta1', 'theta2', 'f12', 'pg1', 'pg2', 'pl1', 'pl2']

idx_nl = pd.to_timedelta(t_nl, unit='s')

df_X_simul = pd.DataFrame(y_nl.T, columns=STATE_COLS, index=idx_nl)

df_Y_simul = pd.DataFrame({
    'theta1': y_nl[0],
    'theta2': y_nl[3],
    'f12'   : F12_nl,
    'pg1'   : P1L + F12_nl,
    'pg2'   : P2L - F12_nl,
    'pl1'   : np.full(N_NL, P1L),
    'pl2'   : np.full(N_NL, P2L),
}, index=idx_nl)

print("Nonlinear DataFrames:")
print(f"  df_X_simul : {df_X_simul.shape}  (7 states)")
print(f"  df_Y_simul : {df_Y_simul.shape}  (7 outputs)")
print(f"\nValue of F12 just before line trip : {F12_nl[t_nl < T_BREAK][-1]:.2f} MW")
print(f"Value of F12 just after  line trip : {F12_nl[t_nl >= T_BREAK][0]:.2f} MW")

In [ ]:
def vline_trip(fig):
    """Add a vertical dashed line at the line trip instant."""
    fig.add_vline(x=T_BREAK, line_dash='dot', line_color='orange', line_width=2,
                  annotation_text=f' Line trip (t={T_BREAK} s, KL→KL/2)',
                  annotation_position='top right')

# ── Frequency ─────────────────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_nl, y=y_nl[1]/(2*np.pi), name='ω₁/2π  (G1)',
                         line=dict(color='royalblue', width=1.8)))
fig.add_trace(go.Scatter(x=t_nl, y=y_nl[4]/(2*np.pi), name='ω₂/2π  (G2)',
                         line=dict(color='tomato', width=1.8)))
fig.add_hline(y=50.0, line_dash='dash', line_color='grey')
vline_trip(fig)
fig.update_layout(title='Nonlinear Simulation — Frequency Response',
                  xaxis_title='Time [s]', yaxis_title='Frequency [Hz]',
                  template='plotly_white', hovermode='x unified', height=380)
fig.show()

# ── Rotor angles ──────────────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_nl, y=y_nl[0], name='θ₁',
                         line=dict(color='royalblue', width=1.8)))
fig.add_trace(go.Scatter(x=t_nl, y=y_nl[3], name='θ₂',
                         line=dict(color='tomato', width=1.8)))
vline_trip(fig)
fig.update_layout(title='Nonlinear Simulation — Rotor Angles',
                  xaxis_title='Time [s]', yaxis_title='Angle [rad]',
                  template='plotly_white', hovermode='x unified', height=380)
fig.show()

# ── Tie-line power ────────────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_nl, y=F12_nl, name='F₁₂ (nonlinear)',
                         line=dict(color='darkorange', width=1.8)))
vline_trip(fig)
fig.update_layout(title='Nonlinear Simulation — Tie-line Power F₁₂',
                  xaxis_title='Time [s]', yaxis_title='Power [MW]',
                  template='plotly_white', hovermode='x unified', height=380)
fig.show()

# ── Generator power demand ────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_nl, y=P1L + F12_nl, name='PG1 = PL1 + F₁₂',
                         line=dict(color='royalblue', width=1.8)))
fig.add_trace(go.Scatter(x=t_nl, y=P2L - F12_nl, name='PG2 = PL2 − F₁₂',
                         line=dict(color='tomato', width=1.8)))
vline_trip(fig)
fig.update_layout(title='Nonlinear Simulation — Generator Power Demand',
                  xaxis_title='Time [s]', yaxis_title='Power [MW]',
                  template='plotly_white', hovermode='x unified', height=380)
fig.show()

---
## 3. Linearization — Pre-fault and Post-fault Models

Two **separate linearisations** are performed, both anchored at the **pre-fault equilibrium** $\mathbf{x}_{eq}$:

| Model | $K_L$ | Active period |
|-------|-------|--------------|
| Pre-fault | $3064\;\text{MW/rad}$ | $t < 2\,\text{s}$ |
| Post-fault | $1532\;\text{MW/rad}$ | $t \geq 2\,\text{s}$ |

### Extended state vector (9 states)

$$\mathbf{x} = [\theta_1,\; \omega_1,\; T_{m1},\; \theta_2,\; \omega_2,\; T_{m2},\; N,\; p_{l1},\; p_{l2}]^\top$$

The augmented load states satisfy $\dot{p}_{li} = 0$ (constant loads).

### Post-fault bias $\mathbf{b}$

Because $\mathbf{x}_{eq}$ (pre-fault) is **not** an equilibrium of the post-fault dynamics, the Taylor expansion around it introduces a constant offset:

$$\dot{\mathbf{x}} \approx A_{\text{post}}\,(\mathbf{x} - \mathbf{x}_{eq}) + \underbrace{f_{\text{post}}(\mathbf{x}_{eq})}_{\mathbf{b} \neq \mathbf{0}}$$

This bias is computed numerically and added to the **predict** step of the Kalman filter.

In [ ]:
def build_linear_model(params: dict):
    """
    Linearise the power system around the operating equilibrium.

    Returns
    -------
    X_sym : SymPy state vector (9 x 1)
    Y_sym : SymPy output vector (7 x 1)
    A, B, C, D, E : numeric numpy arrays
    """
    # ── Symbolic variables ────────────────────────────────────────────────────
    t1, w1, tm1, t2, w2, tm2, n = sp.symbols('theta1 omega1 Tm1 theta2 omega2 Tm2 N')
    p10, p20, pl1, pl2          = sp.symbols('P10 P20 P1L P2L')
    kl, w0, j1, j2              = sp.symbols('KL omega0 J1 J2')
    d1, d2, a1, a2, b1, b2      = sp.symbols('D1 D2 alpha1 alpha2 beta1 beta2')
    pr1, pr2, ks                = sp.symbols('Pr1 Pr2 Ks')

    X_sym = sp.Matrix([t1, w1, tm1, t2, w2, tm2, n, pl1, pl2])

    # ── Linearised dynamics (sin ≈ θ₁ − θ₂ near equilibrium) ─────────────────
    f12 = kl * (t1 - t2)
    pg1 = pl1 + f12
    pg2 = pl2 - f12

    f = sp.Matrix([
        w1 - w0,
        (tm1 - pg1/w0 - d1*(w1 - w0)) / j1,
        -a1*(tm1*w0 - (p10 + n*pr1)) - b1*(w1 - w0),
        w2 - w0,
        (tm2 - pg2/w0 - d2*(w2 - w0)) / j2,
        -a2*(tm2*w0 - (p20 + n*pr2)) - b2*(w2 - w0),
        -ks*((j1*w1 + j2*w2)/(j1 + j2) - w0),
        sp.Integer(0),
        sp.Integer(0),
    ])

    Y_sym = sp.Matrix([t1, t2, f12, pg1, pg2, pl1, pl2])

    # ── Jacobians ─────────────────────────────────────────────────────────────
    A_sym = f.jacobian(X_sym)
    B_sym = f.jacobian(sp.Matrix([p10, p20]))
    D_sym = f.jacobian(sp.Matrix([pl1, pl2]))
    C_sym = Y_sym.jacobian(X_sym)
    E_sym = Y_sym.jacobian(sp.Matrix([pl1, pl2]))

    # ── Numerical substitution ────────────────────────────────────────────────
    subs = {
        w0: params['omega0'], kl: params['KL'], ks: params['Ks'],
        p10: params['g1']['P0'],  pl1: params['g1']['PL'],
        j1:  params['g1']['J'],   d1:  params['g1']['D'],
        a1:  params['g1']['alpha'], b1: params['g1']['beta'],
        pr1: params['g1']['Pr'],
        p20: params['g2']['P0'],  pl2: params['g2']['PL'],
        j2:  params['g2']['J'],   d2:  params['g2']['D'],
        a2:  params['g2']['alpha'], b2: params['g2']['beta'],
        pr2: params['g2']['Pr'],
    }

    to_num = lambda M: np.array(M.subs(subs)).astype(np.float64)

    return X_sym, Y_sym, to_num(A_sym), to_num(B_sym), to_num(C_sym),            to_num(D_sym), to_num(E_sym)

In [ ]:
# ── Pre-fault model  (KL = 3064 MW/rad) ──────────────────────────────────────
print("Building pre-fault linear model  (KL = 3064 MW/rad) ...")
X_sym, Y_sym, A_pre, B_pre, C_pre, D_pre, E_pre = build_linear_model(PARAMS)
E_pre = np.zeros_like(E_pre)

# ── Post-fault model (KL = 1532 MW/rad) ──────────────────────────────────────
print("Building post-fault linear model (KL = 1532 MW/rad) ...")
PARAMS_post = {**PARAMS, 'KL': PARAMS['KL_post']}
_, _, A_post, B_post, C_post, D_post, E_post = build_linear_model(PARAMS_post)
E_post = np.zeros_like(E_post)

n_x = A_pre.shape[0]   # 9
n_y = C_pre.shape[0]   # 7

print(f"\nState dimension  : {n_x}")
print(f"Output dimension : {n_y}")

# ── 9-state pre-fault equilibrium ─────────────────────────────────────────────
x_eq_9 = np.array([
    0.0,
    w0,
    PARAMS['g1']['P0'] / w0,
    theta2_eq,
    w0,
    PARAMS['g2']['P0'] / w0,
    0.0,
    PARAMS['g1']['PL'],    # pl1 augmented state
    PARAMS['g2']['PL'],    # pl2 augmented state
])

# ── Reference output vectors ──────────────────────────────────────────────────
y_eq_pre  = C_pre  @ x_eq_9   # (7,)
y_eq_post = C_post @ x_eq_9   # (7,)  different because C has KL in it

print(f"\nEquilibrium outputs:")
print(f"  F12 (pre-fault model)  : {y_eq_pre[2]:.4f} MW")
print(f"  F12 (post-fault model) : {y_eq_post[2]:.4f} MW  (uses KL_post in C)")

# ── Post-fault bias vector b = f_post(x_eq) ───────────────────────────────────
# x_eq_9[:7] is the 7-state equilibrium; evaluate the post-fault ODE there.
b_post_7 = np.array(power_system_ode(T_BREAK + 1e-9, x_eq_9[:7], PARAMS, KL_post))
b_post_9 = np.append(b_post_7, [0.0, 0.0])   # augmented loads: constant

labels = ['dθ₁', 'dω₁', 'dTm1', 'dθ₂', 'dω₂', 'dTm2', 'dN', 'dpl1', 'dpl2']
print("\nPost-fault bias f_post(x_eq)  [non-zero entries only]:")
for lbl, val in zip(labels, b_post_9):
    if abs(val) > 1e-12:
        print(f"  {lbl:6s} : {val:.6f}")

print("\nMatrix A_pre (first 3 rows):\n",  np.round(A_pre[:3], 4))
print("\nMatrix A_post (first 3 rows):\n", np.round(A_post[:3], 4))

---
## 4. Observability Analysis

$$\mathcal{O} = \begin{bmatrix} C \\ CA \\ CA^2 \\ \vdots \\ CA^{n-1} \end{bmatrix} \in \mathbb{R}^{(n_y \cdot n) \times n}$$

Both models (pre-fault and post-fault) are checked.

In [ ]:
def check_observability(A: np.ndarray, C: np.ndarray, tol: float = 1e-8) -> dict:
    n  = A.shape[0]
    O  = np.vstack([C @ np.linalg.matrix_power(A, k) for k in range(n)])
    rk = np.linalg.matrix_rank(O, tol=tol)
    return {'O': O, 'rank': rk, 'n': n, 'observable': rk == n}

for label, A, C in [('Pre-fault  (KL = 3064)', A_pre, C_pre),
                     ('Post-fault (KL = 1532)', A_post, C_post)]:
    obs = check_observability(A, C)
    status = 'FULLY OBSERVABLE ✓' if obs['observable'] else f'NOT fully observable — {obs["n"]-obs["rank"]} unobservable mode(s)'
    print(f"{label}:  rank = {obs['rank']} / {obs['n']}  →  {status}")

---
## 5. Linear Model Simulation (Euler Forward, Switching)

The linearised model is integrated with the **Euler forward** method.  
The A matrix and output matrix C switch at $t = 2\,\text{s}$.

$$\Delta\mathbf{x}_{k+1} = \Delta\mathbf{x}_k + \bigl(A(t)\,\Delta\mathbf{x}_k + \mathbf{b}(t)\bigr)\,\Delta t$$

where:
- $A(t) = A_{\text{pre}}$ and $\mathbf{b}(t) = \mathbf{0}$ for $t < 2\,\text{s}$
- $A(t) = A_{\text{post}}$ and $\mathbf{b}(t) = f_{\text{post}}(\mathbf{x}_{eq})$ for $t \geq 2\,\text{s}$

> **Note:** $\Delta t = 10^{-5}\,\text{s}$ is required for stability of the Euler scheme given the stiff eigenvalue $\lambda \approx -\alpha_1\omega_0 \approx -31\,416\,\text{s}^{-1}$.

In [ ]:
DT_LIN  = 1e-5      # Euler time step  [s]  — must be < 3.2e-5 for stability
T_LIN   = 20.0
t_steps = np.arange(0.0, T_LIN, DT_LIN)
N_LIN   = len(t_steps)
i_break_lin = int(T_BREAK / DT_LIN)   # step index of the line trip

x_lin_hist = np.empty((N_LIN, n_x))
y_lin_hist = np.empty((N_LIN, n_y))

delta_x = np.zeros(n_x)   # start at equilibrium → deviation = 0
x_lin_hist[0] = x_eq_9
y_lin_hist[0] = C_pre @ x_eq_9

print(f"Euler integration: {N_LIN:,} steps at dt = {DT_LIN} s")
print(f"Model switch at step {i_break_lin:,}  (t = {T_BREAK} s)")

for i in tqdm(range(1, N_LIN)):
    if i <= i_break_lin:
        # Pre-fault: b = 0 (x_eq is a true equilibrium)
        ddx   = A_pre @ delta_x
        C_cur = C_pre
    else:
        # Post-fault: include bias b_post (x_eq is NOT an equilibrium here)
        ddx   = A_post @ delta_x + b_post_9
        C_cur = C_post

    delta_x      = delta_x + ddx * DT_LIN
    x_abs        = delta_x + x_eq_9
    x_lin_hist[i] = x_abs
    y_lin_hist[i] = C_cur @ x_abs

STATE_COLS_9  = ['theta1', 'omega1', 'Tm1', 'theta2', 'omega2', 'Tm2', 'N', 'pl1', 'pl2']

df_X_lin = pd.DataFrame(x_lin_hist, columns=STATE_COLS_9,
                         index=pd.to_timedelta(t_steps, unit='s'))
df_Y_lin = pd.DataFrame(y_lin_hist, columns=OUTPUT_COLS,
                         index=pd.to_timedelta(t_steps, unit='s'))

print("\nLinear simulation complete.")
print(f"  df_X_lin : {df_X_lin.shape}  |  df_Y_lin : {df_Y_lin.shape}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Linear Simulation (Euler) — Line Trip at t = 2 s', fontsize=13, fontweight='bold')

axes[0, 0].plot(t_steps, df_X_lin['omega1'] / (2*np.pi), label='G1 (linear)')
axes[0, 0].plot(t_nl,    y_nl[1] / (2*np.pi), '--', label='G1 (nonlinear)', alpha=0.6)
axes[0, 0].axvline(T_BREAK, color='orange', linestyle=':', label='Line trip')
axes[0, 0].set_title('Frequency G1');  axes[0, 0].set_ylabel('Hz');  axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(t_steps, df_Y_lin['theta1'], label='θ₁ linear')
axes[0, 1].plot(t_steps, df_Y_lin['theta2'], label='θ₂ linear')
axes[0, 1].plot(t_nl, y_nl[0], '--', alpha=0.5, label='θ₁ nonlin')
axes[0, 1].plot(t_nl, y_nl[3], '--', alpha=0.5, label='θ₂ nonlin')
axes[0, 1].axvline(T_BREAK, color='orange', linestyle=':')
axes[0, 1].set_title('Rotor Angles');  axes[0, 1].set_ylabel('rad');  axes[0, 1].legend(fontsize=8)

axes[1, 0].plot(t_steps, df_Y_lin['f12'], label='F₁₂ linear')
axes[1, 0].plot(t_nl, F12_nl, '--', alpha=0.6, label='F₁₂ nonlinear')
axes[1, 0].axvline(T_BREAK, color='orange', linestyle=':')
axes[1, 0].set_title('Tie-line Power F₁₂');  axes[1, 0].set_ylabel('MW');  axes[1, 0].legend(fontsize=8)

axes[1, 1].plot(t_steps, df_Y_lin['pg1'], label='PG1 linear')
axes[1, 1].plot(t_steps, df_Y_lin['pg2'], label='PG2 linear')
axes[1, 1].axvline(T_BREAK, color='orange', linestyle=':')
axes[1, 1].set_title('Generator Power');  axes[1, 1].set_ylabel('MW');  axes[1, 1].legend(fontsize=8)

for ax in axes.flat:
    ax.set_xlabel('Time [s]')
plt.tight_layout()
plt.show()

---
## 6. Kalman Filter

### Discretisation

Continuous-time matrices are discretised via the **matrix exponential** at step $\Delta t_K = 10^{-4}\,\text{s}$:

$$A_d = e^{A\,\Delta t_K}$$

Two pairs of discrete matrices are pre-computed: $(A_d^{\text{pre}},\, C^{\text{pre}})$ and $(A_d^{\text{post}},\, C^{\text{post}})$.

### Filter equations

**Predict** (with bias correction for $t \geq 2\,\text{s}$):

$$\hat{\mathbf{x}}^-_k = A_d(t)\,\hat{\mathbf{x}}_{k-1} + \mathbf{b}_d(t), \qquad P^-_k = A_d P_{k-1} A_d^\top + Q$$

where $\mathbf{b}_d(t) = f_{\text{post}}(\mathbf{x}_{eq})\,\Delta t_K$ for $t \geq t_{\text{break}}$, $\mathbf{0}$ otherwise.

**Correct:**

$$S_k = C(t)\,P^-_k\,C(t)^\top + R, \qquad K_k = P^-_k\,C(t)^\top S_k^{-1}$$
$$\hat{\mathbf{x}}_k = \hat{\mathbf{x}}^-_k + K_k\bigl(\Delta\mathbf{y}_k - C(t)\hat{\mathbf{x}}^-_k\bigr), \qquad P_k = (I - K_k C(t))\,P^-_k$$

### Inputs

Noisy measurements are generated from the **nonlinear simulation** (ground truth):
$\mathbf{y}_{\text{noisy}} = \mathbf{y}_{\text{nl}} + \eta$, $\eta \sim \mathcal{N}(0, R)$, with $\sigma = 1\%$ of signal amplitude per channel.

In [ ]:
# ── Discretisation ────────────────────────────────────────────────────────────
DT_K = 1e-4   # Kalman filter step [s]

Ad_pre  = expm(A_pre  * DT_K)
Ad_post = expm(A_post * DT_K)
Bd_pre  = B_pre  * DT_K
Bd_post = B_post * DT_K

# Discretised bias vector (column vector, 9 x 1)
b_post_d = (b_post_9 * DT_K).reshape(-1, 1)

print("Discrete eigenvalues (Ad_pre)  — max |λ|:", np.max(np.abs(np.linalg.eigvals(Ad_pre))))
print("Discrete eigenvalues (Ad_post) — max |λ|:", np.max(np.abs(np.linalg.eigvals(Ad_post))))

# ── Resample nonlinear outputs to Kalman time grid ────────────────────────────
df_Y_k = df_Y_simul.resample(f'{DT_K}s').interpolate(method='linear')
t_K    = df_Y_k.index.total_seconds().values
N_K    = len(t_K)
i_break_K = int(T_BREAK / DT_K)   # step index at which to switch model

print(f"\nKalman grid: {N_K:,} steps  |  switch at step {i_break_K:,}")

# ── Measurement noise (1 % of max signal amplitude per channel) ───────────────
sigma_meas = np.abs(0.01 * df_Y_k.abs().max().values)
sigma_meas = np.where(sigma_meas < 1e-12, 1e-6, sigma_meas)

rng = np.random.default_rng(42)
df_Y_noisy = df_Y_k.copy()
df_Y_noisy += rng.normal(0.0, sigma_meas, df_Y_k.shape)

print("\nMeasurement noise σ per channel:")
for col, s in zip(OUTPUT_COLS, sigma_meas):
    print(f"  {col:8s}: {s:.4e}")

# ── Noise covariance matrices ─────────────────────────────────────────────────
Q_base = np.diag([
    1e-6,   # θ₁  [rad²]
    1e-6,   # ω₁  [rad²/s²]
    1e-4,   # Tm1
    1e-6,   # θ₂
    1e-6,   # ω₂
    1e-4,   # Tm2
    1e-6,   # N
    1e-4,  # pl1 — constant load, very small process noise
    1e-4,  # pl2 — constant load, very small process noise
])

R_k = np.diag(sigma_meas**2)

# ── Reference output vectors (column) ────────────────────────────────────────
y_eq_pre_c  = y_eq_pre.reshape(-1, 1)
y_eq_post_c = y_eq_post.reshape(-1, 1)

In [ ]:
# ── Initialisation ────────────────────────────────────────────────────────────
x_hat  = np.zeros((n_x, 1))    # delta_x = 0  (start at equilibrium)
P_cov  = np.eye(n_x) * 0.1

xhat_hist = np.empty((N_K, n_x))
yhat_hist = np.empty((N_K, n_y))
trace_P   = []

du = np.zeros((2, 1))   # no setpoint change
eig_list = []

# ── Filter loop ───────────────────────────────────────────────────────────────
print(f"Running Kalman filter: {N_K:,} steps (model switch at step {i_break_K:,}) ...")
for i in tqdm(range(N_K)):

    # ── Select model based on current time ────────────────────────────────────
    if i < i_break_K:
        Ad_k  = Ad_pre
        C_k   = C_pre
        y_eq  = y_eq_pre_c
        bias  = np.zeros((n_x, 1))       # b = 0 pre-fault (at true equilibrium)
    else:
        Ad_k  = Ad_post
        C_k   = C_post
        y_eq  = y_eq_post_c
        bias  = b_post_d                  # b ≠ 0 post-fault
        
    if i == i_break_K:
        print(f"\nModel switch at t = {t_K[i]:.2f} s  (step {i})")
        P_cov *= 10.0   # increase uncertainty at the model switch instant

    # ── Innovation (deviation measurement) ───────────────────────────────────
    delta_y = df_Y_noisy.iloc[i].values.reshape(-1, 1) - y_eq

    # ── Predict ───────────────────────────────────────────────────────────────
    x_hat_minus = Ad_k @ x_hat + bias
    P_minus     = Ad_k @ P_cov @ Ad_k.T + Q_base

    # ── Correct ───────────────────────────────────────────────────────────────
    S           = C_k @ P_minus @ C_k.T + R_k
    K           = P_minus @ C_k.T @ np.linalg.inv(S)
    innovation  = delta_y - C_k @ x_hat_minus
    x_hat       = x_hat_minus + K @ innovation
    P_cov       = (np.eye(n_x) - K @ C_k) @ P_minus

    xhat_hist[i] = x_hat.flatten()
    yhat_hist[i] = (C_k @ x_hat + y_eq).flatten()
    trace_P.append(np.trace(P_cov))
    eig_list.append(np.linalg.eigvals(P_cov))

print("Kalman filter complete.")
print(f"Final covariance trace: {trace_P[-1]:.4e}")
P_eigvalues = pd.DataFrame(eig_list)
# ── Build output DataFrames ───────────────────────────────────────────────────
STATE_COLS_9 = ['theta1', 'omega1', 'Tm1', 'theta2', 'omega2', 'Tm2', 'N', 'pl1', 'pl2']

df_X_hat = pd.DataFrame(xhat_hist + x_eq_9, columns=STATE_COLS_9)
df_X_hat['omega1'] /= (2 * np.pi)
df_X_hat['omega2'] /= (2 * np.pi)
df_X_hat['time']   = t_K

df_Y_hat = pd.DataFrame(yhat_hist, columns=OUTPUT_COLS)
df_Y_hat['time'] = t_K

# ── Covariance trace ──────────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_K[5:], y=trace_P[5:], line=dict(color='teal', width=1.5)))
fig.add_vline(x=T_BREAK, line_dash='dot', line_color='orange',
              annotation_text='Line trip', annotation_position='top right')
fig.update_layout(title='Trace of Estimation Error Covariance P over Time',
                  xaxis_title='Time [s]', yaxis_title='trace(P)',
                  template='plotly_white', height=320)
fig.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. Convert the list to a DataFrame after the loop
# Assuming 'eig_list' was populated during the loop
P_eigvalues_df = pd.DataFrame(eig_list)

# 2. Create the figure
fig, axes = plt.subplots(1, figsize=(14, 8))
fig.suptitle('Covariance Analysis', fontsize=14, fontweight='bold')

# 3. Plot the data 
axes.plot(t_K, P_eigvalues_df)

axes.set_title('Eigenvalues of $P_{cov}$')  
axes.set_xlabel('Time [s]')
axes.set_ylabel('Eigenvalue')

# 4. Set custom legend labels matching the 9 states
custom_labels = [
    r'$\lambda_{\theta_1}$', 
    r'$\lambda_{\omega_1}$', 
    r'$\lambda_{T_{m1}}$', 
    r'$\lambda_{\theta_2}$', 
    r'$\lambda_{\omega_2}$', 
    r'$\lambda_{T_{m2}}$', 
    r'$\lambda_{N}$', 
    r'$\lambda_{P_{L1}}$', 
    r'$\lambda_{P_{L2}}$'
]
axes.legend(custom_labels) 

# 5. Restrict the x-axis to plot only between 0 and 5 seconds
axes.set_xlim([0, 5])

axes.set_yscale('log')
axes.grid(True, which="both", ls="-", alpha=0.5)
plt.tight_layout()
plt.show()

---
## 7. Results & Visualisation

Each plot shows three curves:
- 🟢 **Nonlinear simulation** — ground truth
- ⚪ **Noisy measurement** — simulated sensor
- 🔴 **Kalman estimate** — filter output

The orange dotted line marks the **line trip at $t = 2\,\text{s}$**.

In [ ]:
def compute_snr(signal: np.ndarray, estimate: np.ndarray) -> float:
    """SNR [dB] = 10 log10(var(signal) / var(signal − estimate))."""
    noise_power  = np.var(signal - estimate)
    signal_power = np.var(signal)
    if noise_power < 1e-30:
        return np.inf
    return 10 * np.log10(signal_power / noise_power)


def plot_kalman(time, y_nl_vals, y_noisy_vals, y_hat_vals, title, ylabel, snr=None):
    """Plot nonlinear ground truth / noisy / Kalman estimate."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=time, y=y_hat_vals, name='Kalman',
                             line=dict(color='red', width=1.8)))
    
    fig.add_trace(go.Scatter(x=time, y=y_noisy_vals, name='Noisy',
                             line=dict(color='grey', width=1.0)))
    
    fig.add_trace(go.Scatter(x=time, y=y_nl_vals, name='Nonlinear simulation',
                             line=dict(color='green', width=1.8)))
    
    fig.add_vline(x=T_BREAK, line_dash='dot', line_color='orange',
                  annotation_text=' Line trip', annotation_position='top right')
    snr_txt = f'  |  SNR = {snr:.1f} dB' if snr is not None else ''
    fig.update_layout(
        title=dict(text=title + snr_txt, font=dict(size=13)),
        xaxis_title='Time [s]', yaxis_title=ylabel,
        template='plotly_white', hovermode='x unified',
        legend=dict(orientation='h', y=1.12), height=360,
    )
    fig.show()

# Resample nonlinear to Kalman grid for comparison
df_NL_k = df_Y_simul.resample(f'{DT_K}s').interpolate(method='linear')

In [ ]:
# ── Tie-line power F12 ────────────────────────────────────────────────────────
snr_f12 = compute_snr(df_NL_k['f12'].values, df_Y_hat['f12'].values)

plot_kalman(
    time       = t_K,
    y_nl_vals  = df_NL_k['f12'].values,
    y_noisy_vals = df_Y_noisy['f12'].values,
    y_hat_vals = df_Y_hat['f12'].values,
    title='Tie-line Power F₁₂',
    ylabel='Flux [MW]',
    snr=snr_f12,
)

In [ ]:
# ── Generated power G1 ────────────────────────────────────────────────────────
snr_pg1 = compute_snr(df_NL_k['pg1'].values, df_Y_hat['pg1'].values)

plot_kalman(
    time       = t_K,
    y_nl_vals  = df_NL_k['pg1'].values,
    y_noisy_vals = df_Y_noisy['pg1'].values,
    y_hat_vals = df_Y_hat['pg1'].values,
    title='Generator Power Demand PG1',
    ylabel='Power [MW]',
    snr=snr_pg1,
)

In [ ]:
# ── Load power G1 ────────────────────────────────────────────────────────
snr_pg1 = compute_snr(df_NL_k['pl1'].values, df_Y_hat['pl1'].values)

plot_kalman(
    time       = t_K,
    y_nl_vals  = df_NL_k['pl1'].values,
    y_noisy_vals = df_Y_noisy['pl1'].values,
    y_hat_vals = df_Y_hat['pl1'].values,
    title='Load Power Demand PL1',
    ylabel='Load [MW]',
    snr=snr_pg1,
)

In [ ]:
# ── Generated power G2 ────────────────────────────────────────────────────────
snr_pg2 = compute_snr(df_NL_k['pg2'].values, df_Y_hat['pg2'].values)

plot_kalman(
    time       = t_K,
    y_nl_vals  = df_NL_k['pg2'].values,
    y_noisy_vals = df_Y_noisy['pg2'].values,
    y_hat_vals = df_Y_hat['pg2'].values,
    title='Generator Power Demand PG2',
    ylabel='Power [MW]',
    snr=snr_pg2,
)

In [ ]:
# ── Angular frequency G1 ─────────────────────────────────────────────────────
df_X_NL_k = df_X_simul.resample(f'{DT_K}s').interpolate(method='linear')

snr_w1 = compute_snr(
    df_X_NL_k['omega1'].values / (2*np.pi),
    df_X_hat['omega1'].values,
)

plot_kalman(
    time       = t_K,
    y_nl_vals  = df_X_NL_k['omega1'].values / (2*np.pi),
    y_noisy_vals = df_X_NL_k['omega1'].values / (2*np.pi)
                   + rng.standard_normal(N_K) * sigma_meas[1] / (2*np.pi),
    y_hat_vals = df_X_hat['omega1'].values,
    title='Angular Frequency ω₁ (G1)',
    ylabel='Frequency [Hz]',
    snr=snr_w1,
)

In [ ]:
# ── Rotor angle θ₁ ────────────────────────────────────────────────────────────
snr_th1 = compute_snr(df_X_NL_k['theta1'].values, df_Y_hat['theta1'].values)

plot_kalman(
    time       = t_K,
    y_nl_vals  = df_X_NL_k['theta1'].values,
    y_noisy_vals = df_Y_noisy['theta1'].values,
    y_hat_vals = df_Y_hat['theta1'].values,
    title='Rotor Angle θ₁ (G1)',
    ylabel='Angle [rad]',
    snr=snr_th1,
)

# ── Rotor angle θ₂ ────────────────────────────────────────────────────────────
snr_th2 = compute_snr(df_X_NL_k['theta2'].values, df_Y_hat['theta2'].values)

plot_kalman(
    time       = t_K,
    y_nl_vals  = df_X_NL_k['theta2'].values,
    y_noisy_vals = df_Y_noisy['theta2'].values,
    y_hat_vals = df_Y_hat['theta2'].values,
    title='Rotor Angle θ₂ (G2)',
    ylabel='Angle [rad]',
    snr=snr_th2,
)

In [ ]:
# ── Secondary control signal N ────────────────────────────────────────────────
snr_N = compute_snr(df_X_NL_k['N'].values, df_X_hat['N'].values)

plot_kalman(
    time       = t_K,
    y_nl_vals  = df_X_NL_k['N'].values,
    y_noisy_vals = df_X_NL_k['N'].values + rng.standard_normal(N_K) * 1e-4,
    y_hat_vals = df_X_hat['N'].values,
    title='Secondary Control Signal N',
    ylabel='p.u.',
    snr=snr_N,
)

In [ ]:
# ── Augmented load state estimation ──────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Augmented Load State Estimation (constant loads)', fontsize=13, fontweight='bold')

ax1.plot(t_K, df_X_hat['pl1'].values, color='crimson', label='Kalman pl1', lw=1.5)
ax1.axhline(PARAMS['g1']['PL'], color='royalblue', linestyle='--', label=f'True PL1 = {PARAMS["g1"]["PL"]} MW')
ax1.axvline(T_BREAK, color='orange', linestyle=':', label='Line trip')
ax1.set_title('Load PL1'); ax1.set_xlabel('Time [s]'); ax1.set_ylabel('MW'); ax1.legend()

ax2.plot(t_K, df_X_hat['pl2'].values, color='crimson', label='Kalman pl2', lw=1.5)
ax2.axhline(PARAMS['g2']['PL'], color='tomato', linestyle='--', label=f'True PL2 = {PARAMS["g2"]["PL"]} MW')
ax2.axvline(T_BREAK, color='orange', linestyle=':')
ax2.set_title('Load PL2'); ax2.set_xlabel('Time [s]'); ax2.set_ylabel('MW'); ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── SNR Summary — relative error (%) ─────────────────────────────────────────
def rel_err_pct(truth, estimate):
    return (np.std(estimate - truth) / max(np.abs(np.mean(truth)), 1e-6)) * 100

snr_summary = {
    'F12'    : (compute_snr(df_NL_k['f12'].values,     df_Y_hat['f12'].values),
                rel_err_pct(df_NL_k['f12'].values,     df_Y_hat['f12'].values)),
    'PG1'    : (compute_snr(df_NL_k['pg1'].values,     df_Y_hat['pg1'].values),
                rel_err_pct(df_NL_k['pg1'].values,     df_Y_hat['pg1'].values)),
    'PG2'    : (compute_snr(df_NL_k['pg2'].values,     df_Y_hat['pg2'].values),
                rel_err_pct(df_NL_k['pg2'].values,     df_Y_hat['pg2'].values)),
    'ω₁'     : (compute_snr(df_X_NL_k['omega1'].values/(2*np.pi), df_X_hat['omega1'].values),
                rel_err_pct(df_X_NL_k['omega1'].values/(2*np.pi), df_X_hat['omega1'].values)),
    'θ₁'     : (compute_snr(df_X_NL_k['theta1'].values, df_Y_hat['theta1'].values),
                rel_err_pct(df_X_NL_k['theta1'].values, df_Y_hat['theta1'].values)),
    'N'      : (compute_snr(df_X_NL_k['N'].values,     df_X_hat['N'].values),
                rel_err_pct(df_X_NL_k['N'].values,     df_X_hat['N'].values)),
}

print("=" * 55)
print(f"  SNR SUMMARY — Line Trip at t = {T_BREAK} s, No Load Disturbance")
print("=" * 55)
print(f"  {'Signal':8s}  {'SNR [dB]':>10s}   {'Rel. Error':>10s}")
print("-" * 55)
for sig, (snr_val, err) in snr_summary.items():
    snr_str = f'{snr_val:8.1f}' if np.isfinite(snr_val) else '      ∞'
    print(f"  {sig:8s}  {snr_str}     {err:8.3f} %")
print("=" * 55)

In [ ]:
# ── Calculation Functions ─────────────────────────────────────────────────────
def rel_err_pct(truth, estimate):
    # max(..., 1e-6) prevents division by zero if the signal mean is strictly zero
    return (np.std(estimate - truth) / max(np.abs(np.mean(truth)), 1e-6)) * 100

print("=" * 65)
print(f"  PERFORMANCE SUMMARY — Line Trip at t = {T_BREAK} s")
print("=" * 65)
print(f"  {'Signal':<10s} | {'SNR [dB]':>12s} | {'Rel. Error (%)':>15s}")
print("-" * 65)

snr_summary = {}

# 1. Dynamic loop over all state variables (X)
x_cols = [c for c in df_X_hat.columns if c in df_X_NL_k.columns and c != 'time']
for col in x_cols:
    truth = df_X_NL_k[col].values
    estimate = df_X_hat[col].values
    
    # Specific handling: rad/s -> Hz conversion for rotor speeds
    if 'omega' in col.lower():
        truth = truth / (2 * np.pi)
        
    snr_val = compute_snr(truth, estimate)
    err_val = rel_err_pct(truth, estimate)
    
    # Display formatting
    snr_str = f'{snr_val:10.2f}' if np.isfinite(snr_val) else '         ∞'
    print(f"  X_{col:<8s} | {snr_str} dB | {err_val:13.4f} %")
    
    # Storage (optional, if you need the dictionary later on)
    snr_summary[f"X_{col}"] = (snr_val, err_val)

# 2. Dynamic loop over all output variables (Y)
y_cols = [c for c in df_Y_hat.columns if c in df_NL_k.columns and c != 'time']
for col in y_cols:
    truth = df_NL_k[col].values
    estimate = df_Y_hat[col].values
    
    snr_val = compute_snr(truth, estimate)
    err_val = rel_err_pct(truth, estimate)
    
    # Display formatting
    snr_str = f'{snr_val:10.2f}' if np.isfinite(snr_val) else '         ∞'
    print(f"  Y_{col:<8s} | {snr_str} dB | {err_val:13.4f} %")
    
    # Storage (optional)
    snr_summary[f"Y_{col}"] = (snr_val, err_val)

print("=" * 65)